# Exploring ARC-AGI-1

Load the dataset, visualize tasks, and identify the simplest ones for prototyping.

In [ ]:
import arckit
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

train_set, eval_set = arckit.load_data()
print(f"Training tasks: {len(train_set)}")
print(f"Eval tasks:     {len(eval_set)}")

In [ ]:
ARC_COLORS = [
    '#000000',  # 0: black
    '#0074D9',  # 1: blue
    '#FF4136',  # 2: red
    '#2ECC40',  # 3: green
    '#FFDC00',  # 4: yellow
    '#AAAAAA',  # 5: grey
    '#F012BE',  # 6: magenta
    '#FF851B',  # 7: orange
    '#7FDBFF',  # 8: light blue
    '#870C25',  # 9: maroon
]
ARC_CMAP = ListedColormap(ARC_COLORS)
ARC_NORM = BoundaryNorm(np.arange(-0.5, 10.5, 1), ARC_CMAP.N)


def plot_grid(ax, grid, title=""):
    grid = np.array(grid)
    ax.imshow(grid, cmap=ARC_CMAP, norm=ARC_NORM)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(np.arange(-0.5, grid.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
    ax.grid(which='minor', color='#444', linewidth=0.5)
    ax.tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)


def plot_task(task, max_pairs=5):
    pairs = task.train[:max_pairs] + task.test[:1]
    n = len(pairs)
    fig, axes = plt.subplots(n, 2, figsize=(5, 2.2 * n))
    if n == 1:
        axes = [axes]
    for i, (inp, out) in enumerate(pairs):
        label = f"Train {i+1}" if i < len(task.train[:max_pairs]) else "Test"
        plot_grid(axes[i][0], inp, f"{label} input")
        plot_grid(axes[i][1], out, f"{label} output")
    fig.suptitle(f"Task {task.id}", fontsize=12, fontweight='bold')
    fig.tight_layout()
    plt.show()

In [ ]:
# Show a few random tasks
rng = np.random.default_rng(42)
for idx in rng.choice(len(train_set), size=5, replace=False):
    plot_task(train_set[idx])

In [ ]:
# Gather stats on every task
stats = []
for task in train_set:
    for inp, out in task.train + task.test:
        inp_arr = np.array(inp)
        out_arr = np.array(out)
        stats.append({
            'id': task.id,
            'n_train': len(task.train),
            'n_test': len(task.test),
            'inp_h': inp_arr.shape[0],
            'inp_w': inp_arr.shape[1],
            'out_h': out_arr.shape[0],
            'out_w': out_arr.shape[1],
            'inp_colors': len(np.unique(inp_arr)),
            'out_colors': len(np.unique(out_arr)),
            'same_shape': inp_arr.shape == out_arr.shape,
            'inp_cells': inp_arr.size,
            'out_cells': out_arr.size,
        })

import pandas as pd  # only used in this notebook for convenience
df = pd.DataFrame(stats)
print("=== Grid size distribution ===")
print(f"Input  H: {df.inp_h.describe().to_dict()}")
print(f"Input  W: {df.inp_w.describe().to_dict()}")
print(f"Output H: {df.out_h.describe().to_dict()}")
print(f"Output W: {df.out_w.describe().to_dict()}")
print(f"\nSame input/output shape: {df.same_shape.mean():.1%} of pairs")
print(f"\n=== Colors per grid ===")
print(f"Input colors:  {df.inp_colors.describe().to_dict()}")
print(f"Output colors: {df.out_colors.describe().to_dict()}")
print(f"\n=== Training pairs per task ===")
print(df.groupby('id').first().n_train.value_counts().sort_index())

In [ ]:
# Find the simplest tasks: small grids, same shape in/out, few colors
task_stats = df.groupby('id').agg({
    'inp_cells': 'max',
    'out_cells': 'max',
    'same_shape': 'all',
    'inp_colors': 'max',
    'out_colors': 'max',
    'n_train': 'first',
}).reset_index()

task_stats['total_cells'] = task_stats['inp_cells'] + task_stats['out_cells']
task_stats['max_colors'] = task_stats[['inp_colors', 'out_colors']].max(axis=1)

simple = task_stats[task_stats.same_shape].sort_values('total_cells').head(20)
print("=== 20 simplest same-shape tasks (by total cells) ===")
print(simple[['id', 'inp_cells', 'max_colors', 'n_train']].to_string(index=False))

print("\n--- Visualizing the top 5 ---")
task_by_id = {t.id: t for t in train_set}
for tid in simple.head(5)['id']:
    plot_task(task_by_id[tid])

In [ ]:
# Look for potential "identity / copy" tasks: output == input
identity_tasks = []
for task in train_set:
    is_identity = all(
        np.array_equal(np.array(inp), np.array(out))
        for inp, out in task.train
    )
    if is_identity:
        identity_tasks.append(task)

print(f"Tasks where output == input on all training pairs: {len(identity_tasks)}")
for t in identity_tasks[:3]:
    plot_task(t)

In [ ]:
# Look for tasks where the transformation is a simple color swap
color_swap_tasks = []
for task in train_set:
    all_same_shape = all(
        np.array(inp).shape == np.array(out).shape
        for inp, out in task.train
    )
    if not all_same_shape:
        continue

    # Check if there's a consistent per-color mapping across all pairs
    mapping = {}
    consistent = True
    for inp, out in task.train:
        inp_arr = np.array(inp)
        out_arr = np.array(out)
        for c_in, c_out in zip(inp_arr.flat, out_arr.flat):
            if c_in in mapping:
                if mapping[c_in] != c_out:
                    consistent = False
                    break
            else:
                mapping[c_in] = c_out
        if not consistent:
            break

    if consistent and len(mapping) > 0:
        color_swap_tasks.append((task, mapping))

print(f"Tasks with consistent per-cell color mapping: {len(color_swap_tasks)}")
for t, m in color_swap_tasks[:5]:
    print(f"  {t.id}: {m}")
    plot_task(t)